In [ ]:
import pandas as pd
import requests

TZ      = 'Europe/Luxembourg'
dataset = 'IDCCD_intradayAtc'
day     = pd.Timestamp.now(tz=TZ).normalize() #+ pd.Timedelta(days=1)

r = requests.get(
    f'https://publicationtool.jao.eu/coreID/api/data/{dataset}',
    params={
        'FromUtc': day.tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%S.000Z'),
        'ToUtc':  (day + pd.Timedelta(days=1)).tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%S.000Z'),
    },
    timeout=60,
)
r.raise_for_status()
rows = r.json().get('data', [])
assert rows, f'No data for {day.date()} — IDCC(d) not yet published.'

df = pd.DataFrame(rows)
df['mtu'] = pd.to_datetime(df['dateTimeUtc'], utc=True).dt.tz_convert(TZ)
df = df.set_index('mtu').sort_index()
df = df[[c for c in df.columns if c.startswith('border_')]]
df.columns = [c.replace('border_', '').replace('_', '>') for c in df.columns]
print(f'IDCC(d) today: {df.index.min().strftime("%Y-%m-%d %H:%M")} -> {df.index.max().strftime("%H:%M")}  ({len(df)} rows)')

In [ ]:
cap_15 = pd.DataFrame({
    # CZ neighbours
    'cz_AT_imp': df['AT>CZ'],
    'cz_AT_exp': df['CZ>AT'],
    'cz_DE_imp': df['DE>CZ'],
    'cz_DE_exp': df['CZ>DE'],
    'cz_PL_imp': df['PL>CZ'],
    'cz_PL_exp': df['CZ>PL'],
    'cz_SK_imp': df['SK>CZ'],
    'cz_SK_exp': df['CZ>SK'],
    # CZ <-> HU: best of (CZ-SK-HU) or (CZ-AT-HU)
    'cz_HU_exp': pd.concat([df[['CZ>SK','SK>HU']].min(axis=1), df[['CZ>AT','AT>HU']].min(axis=1)], axis=1).max(axis=1),
    'cz_HU_imp': pd.concat([df[['HU>SK','SK>CZ']].min(axis=1), df[['HU>AT','AT>CZ']].min(axis=1)], axis=1).max(axis=1),
    # SK neighbours
    'sk_CZ_imp': df['CZ>SK'],
    'sk_CZ_exp': df['SK>CZ'],
    'sk_HU_imp': df['HU>SK'],
    'sk_HU_exp': df['SK>HU'],
    'sk_PL_imp': df['PL>SK'],
    'sk_PL_exp': df['SK>PL'],
    # SK <-> DE: best of (SK-CZ-DE), (SK-HU-AT-DE), or (SK-PL-DE)
    'sk_DE_exp': pd.concat([df[['SK>CZ','CZ>DE']].min(axis=1), df[['SK>HU','HU>AT','AT>DE']].min(axis=1), df[['SK>PL','PL>DE']].min(axis=1)], axis=1).max(axis=1),
    'sk_DE_imp': pd.concat([df[['DE>CZ','CZ>SK']].min(axis=1), df[['DE>AT','AT>HU','HU>SK']].min(axis=1), df[['DE>PL','PL>SK']].min(axis=1)], axis=1).max(axis=1),
    # SK <-> AT: best of (SK-CZ-AT), (SK-HU-AT)
    'sk_AT_exp': pd.concat([df[['SK>CZ','CZ>AT']].min(axis=1), df[['SK>HU','HU>AT']].min(axis=1)], axis=1).max(axis=1),
    'sk_AT_imp': pd.concat([df[['AT>CZ','CZ>SK']].min(axis=1), df[['AT>HU', 'HU>SK']].min(axis=1)], axis=1).max(axis=1),
    # HU neighbours
    'hu_AT_imp': df['AT>HU'],
    'hu_AT_exp': df['HU>AT'],
    'hu_HR_imp': df['HR>HU'],
    'hu_HR_exp': df['HU>HR'],
    'hu_RO_imp': df['RO>HU'],
    'hu_RO_exp': df['HU>RO'],
    'hu_SI_imp': df['SI>HU'],
    'hu_SI_exp': df['HU>SI'],
    'hu_SK_imp': df['SK>HU'],
    'hu_SK_exp': df['HU>SK'],
    # HU <-> CZ: best of (HU-SK-CZ) or (HU-AT-CZ)
    'hu_CZ_exp': pd.concat([df[['HU>SK','SK>CZ']].min(axis=1), df[['HU>AT','AT>CZ']].min(axis=1)], axis=1).max(axis=1),
    'hu_CZ_imp': pd.concat([df[['CZ>SK','SK>HU']].min(axis=1), df[['CZ>AT','AT>HU']].min(axis=1)], axis=1).max(axis=1),
    # HU <-> DE: best of (HU-AT-DE), (HU-SK-CZ-DE), (HU-SK-AT-DE), or (HU-SK-PL-DE)
    'hu_DE_exp': pd.concat([df[['HU>AT','AT>DE']].min(axis=1), df[['HU>SK','SK>CZ','CZ>DE']].min(axis=1), df[['HU>SK','SK>PL','PL>DE']].min(axis=1)], axis=1).max(axis=1),
    'hu_DE_imp': pd.concat([df[['DE>AT','AT>HU']].min(axis=1), df[['DE>CZ','CZ>SK','SK>HU']].min(axis=1), df[['DE>PL','PL>SK','SK>HU']].min(axis=1)], axis=1).max(axis=1),
    # HU <-> PL: through SK
    'hu_PL_exp': df[['HU>SK','SK>PL']].min(axis=1),
    'hu_PL_imp': df[['PL>SK','SK>HU']].min(axis=1)
}).round(0)

print(f'15-min capacities ready: {len(cap_15)} rows')

In [ ]:
fmt = lambda v: 'N/A' if pd.isna(v) else (f'({int(v)})' if int(v) < 0 else str(int(v)))
W, BW = 8, 28

print(f'\n{"="*90}\n  15-MINUTE CAPACITIES\n{"="*90}')
for ts, r in cap_15.iterrows():
    cz = [
        f'CZ      {"import":>{W}}  {"export":>{W}}',
        f'{"-"*(6+W+2+W)}',
        f'AT      {fmt(r.cz_AT_imp):>{W}}  {fmt(r.cz_AT_exp):>{W}}',
        f'DE      {fmt(r.cz_DE_imp):>{W}}  {fmt(r.cz_DE_exp):>{W}}',
        f'HU      {fmt(r.cz_HU_imp):>{W}}  {fmt(r.cz_HU_exp):>{W}}',
        f'PL      {fmt(r.cz_PL_imp):>{W}}  {fmt(r.cz_PL_exp):>{W}}',
        f'SK      {fmt(r.cz_SK_imp):>{W}}  {fmt(r.cz_SK_exp):>{W}}',
    ]
    sk = [
        f'SK      {"import":>{W}}  {"export":>{W}}',
        f'{"-"*(6+W+2+W)}',
        f'AT      {fmt(r.sk_AT_imp):>{W}}  {fmt(r.sk_AT_exp):>{W}}',
        f'CZ      {fmt(r.sk_CZ_imp):>{W}}  {fmt(r.sk_CZ_exp):>{W}}',
        f'DE      {fmt(r.sk_DE_imp):>{W}}  {fmt(r.sk_DE_exp):>{W}}',
        f'HU      {fmt(r.sk_HU_imp):>{W}}  {fmt(r.sk_HU_exp):>{W}}',
        f'PL      {fmt(r.sk_PL_imp):>{W}}  {fmt(r.sk_PL_exp):>{W}}',
    ]
    hu = [
        f'HU      {"import":>{W}}  {"export":>{W}}',
        f'{"-"*(6+W+2+W)}',
        f'AT      {fmt(r.hu_AT_imp):>{W}}  {fmt(r.hu_AT_exp):>{W}}',
        f'CZ      {fmt(r.hu_CZ_imp):>{W}}  {fmt(r.hu_CZ_exp):>{W}}',
        f'DE      {fmt(r.hu_DE_imp):>{W}}  {fmt(r.hu_DE_exp):>{W}}',
        f'HR      {fmt(r.hu_HR_imp):>{W}}  {fmt(r.hu_HR_exp):>{W}}',
        f'PL      {fmt(r.hu_PL_imp):>{W}}  {fmt(r.hu_PL_exp):>{W}}',
        f'RO      {fmt(r.hu_RO_imp):>{W}}  {fmt(r.hu_RO_exp):>{W}}',
        f'SI      {fmt(r.hu_SI_imp):>{W}}  {fmt(r.hu_SI_exp):>{W}}',
        f'SK      {fmt(r.hu_SK_imp):>{W}}  {fmt(r.hu_SK_exp):>{W}}',
    ]
    h = max(len(cz), len(sk), len(hu))
    cz += [''] * (h - len(cz))
    sk += [''] * (h - len(sk))
    hu += [''] * (h - len(hu))
    print(f'\n  {ts.strftime("%m-%d %H:%M")}')
    for a, b, c in zip(cz, sk, hu):
        print(f'  {a:<{BW}}  |  {b:<{BW}}  |  {c}')

In [ ]:

cap_h = cap_15.resample('1h').min()

fmt = lambda v: 'N/A' if pd.isna(v) else (f'({int(v)})' if int(v) < 0 else str(int(v)))
W, BW = 8, 28

print(f'\n{"="*90}\n  HOURLY CAPACITIES  (min of 15-min within each hour)\n{"="*90}')
for ts, r in cap_h.iterrows():
    cz = [
        f'CZ      {"import":>{W}}  {"export":>{W}}',
        f'{"-"*(6+W+2+W)}',
        f'AT      {fmt(r.cz_AT_imp):>{W}}  {fmt(r.cz_AT_exp):>{W}}',
        f'DE      {fmt(r.cz_DE_imp):>{W}}  {fmt(r.cz_DE_exp):>{W}}',
        f'HU      {fmt(r.cz_HU_imp):>{W}}  {fmt(r.cz_HU_exp):>{W}}',
        f'PL      {fmt(r.cz_PL_imp):>{W}}  {fmt(r.cz_PL_exp):>{W}}',
        f'SK      {fmt(r.cz_SK_imp):>{W}}  {fmt(r.cz_SK_exp):>{W}}',
    ]
    sk = [
        f'SK      {"import":>{W}}  {"export":>{W}}',
        f'{"-"*(6+W+2+W)}',
        f'AT      {fmt(r.sk_AT_imp):>{W}}  {fmt(r.sk_AT_exp):>{W}}',
        f'CZ      {fmt(r.sk_CZ_imp):>{W}}  {fmt(r.sk_CZ_exp):>{W}}',
        f'DE      {fmt(r.sk_DE_imp):>{W}}  {fmt(r.sk_DE_exp):>{W}}',
        f'HU      {fmt(r.sk_HU_imp):>{W}}  {fmt(r.sk_HU_exp):>{W}}',
        f'PL      {fmt(r.sk_PL_imp):>{W}}  {fmt(r.sk_PL_exp):>{W}}',
    ]
    hu = [
        f'HU      {"import":>{W}}  {"export":>{W}}',
        f'{"-"*(6+W+2+W)}',
        f'AT      {fmt(r.hu_AT_imp):>{W}}  {fmt(r.hu_AT_exp):>{W}}',
        f'CZ      {fmt(r.hu_CZ_imp):>{W}}  {fmt(r.hu_CZ_exp):>{W}}',
        f'DE      {fmt(r.hu_DE_imp):>{W}}  {fmt(r.hu_DE_exp):>{W}}',
        f'HR      {fmt(r.hu_HR_imp):>{W}}  {fmt(r.hu_HR_exp):>{W}}',
        f'PL      {fmt(r.hu_PL_imp):>{W}}  {fmt(r.hu_PL_exp):>{W}}',
        f'RO      {fmt(r.hu_RO_imp):>{W}}  {fmt(r.hu_RO_exp):>{W}}',
        f'SI      {fmt(r.hu_SI_imp):>{W}}  {fmt(r.hu_SI_exp):>{W}}',
        f'SK      {fmt(r.hu_SK_imp):>{W}}  {fmt(r.hu_SK_exp):>{W}}',
    ]
    h = max(len(cz), len(sk), len(hu))
    cz += [''] * (h - len(cz))
    sk += [''] * (h - len(sk))
    hu += [''] * (h - len(hu))
    print(f'\n  {ts.strftime("%m-%d %H:%M")}')
    for a, b, c in zip(cz, sk, hu):
        print(f'  {a:<{BW}}  |  {b:<{BW}}  |  {c}')